# ViCare — Trợ lý hỏi đáp sức khỏe tiếng Việt bằng RAG (Bản cải tiến AI Challenge 24h)

**Repo GitHub chính thức:** `https://github.com/humman0id/vicare-rag`  
**Môi trường chuẩn để Ban giám khảo tái lập:** **Google Colab miễn phí (GPU T4 — 15 GB VRAM)** *(chọn `Runtime → Change runtime type → T4 GPU`)*

**Tóm tắt cải tiến so với Baseline (`ViCare_Bootcamp24h.ipynb`):**
1. **Embedding nâng cấp (`google/embeddinggemma-2`) + đối chứng công bằng với `intfloat/multilingual-e5-small`:**
   - Dùng *selective encoder loading* (`vision_config=None, audio_config=None`) để chỉ nạp nhánh văn bản **271M** tham số (thay vì 740M đa phương thức).
   - **Tương thích an toàn với T4 miễn phí:** vì T4 không hỗ trợ `bfloat16` phần cứng và EmbeddingGemma 2 bị tràn số ở `float16`, hệ thống tự động chọn **`float32`** trên T4 (và tự động bật **`bfloat16`** nếu chạy trên L4/A100).
   - Tự động giải phóng VRAM của model baseline (`e5_embedder.unload()`) trước khi nạp `embeddinggemma-2` để vừa gọn trong 15 GB VRAM của T4 cùng Ollama `qwen2.5:3b`, đồng thời đo VRAM riêng biệt từng model.
   - Sử dụng task prefix bất đối xứng chuẩn của EmbeddingGemma 2: `prompt_name="QuestionAnswering"` cho câu hỏi và `prompt_name="Document"` cho tài liệu.
   - Chạy song song **cả model baseline mặc định (`intfloat/multilingual-e5-small`)** trên cùng 100 câu `dev` (`seed=2026`) để báo cáo công bằng chất lượng + đánh đổi (VRAM, thời gian lập chỉ mục, độ trễ truy vấn, số chiều vector, yêu cầu phần cứng).
2. **Prompt cải tiến & Cưỡng chế trích dẫn `[1]`, `[2]` nghiêm ngặt:**
   - Tách rõ ranh giới DỮ LIỆU vs QUY TẮC để giảm prompt injection.
   - Kiểm tra hậu sinh (`verify_and_clean_citations`): lọc bỏ mọi số nguồn bịa vượt phạm vi, chạy 1 lượt sửa trích dẫn xác định nếu thiếu `[k]`, và từ chối an toàn (*fail-closed*) nếu vẫn không có trích dẫn hợp lệ.
3. **Hiệu chỉnh ngưỡng từ chối (`MIN_SCORE`) theo phân bố thực tế của từng model:**
   - Thang điểm cosine của `embeddinggemma-2` khác `multilingual-e5-small`; notebook tự động hiệu chỉnh ngưỡng trên tập trong kho vs ngoài kho thay vì giữ cứng `0.86`.
4. **Nâng cấp Guardrails cấp cứu tiếng Việt:**
   - Mở rộng từ khóa dấu hiệu nguy hiểm theo nhóm lâm sàng, hỗ trợ không dấu và nhận diện phủ định (ví dụ *"không đau ngực"* không báo động giả).

> ⚠️ **Giới hạn an toàn:** ViCare chỉ là sản phẩm DEMO học thuật trên ViHealthQA, **không phải** công cụ chẩn đoán, cấp cứu hay tư vấn điều trị. Không nhập thông tin cá nhân nhận dạng người thật.


## 0. Kiểm tra GPU Google Colab (T4 miễn phí) & Cài đặt môi trường


In [ ]:
import os, sys, torch

IN_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG")) or "google.colab" in sys.modules
print("IN_COLAB:", IN_COLAB)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    bf16_ok = torch.cuda.is_bf16_supported()
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU: {gpu_name} ({total_gb:.1f} GB VRAM) | BF16 supported: {bf16_ok}")
    if not bf16_ok:
        print("ℹ️ GPU hiện tại (ví dụ T4 miễn phí) không hỗ trợ BF16 phần cứng → EmbeddingGemma 2 sẽ tự chạy float32 an toàn (không dùng float16).")
else:
    print("⚠️ Chưa bật GPU. Trên Google Colab, chọn Runtime -> Change runtime type -> T4 GPU.")


In [ ]:
# Cài thư viện tương thích Google EmbeddingGemma 2 (sentence-transformers>=6.1, transformers>=5.18)
# và gói vicare trực tiếp từ repo GitHub của dự án.
%pip install -q -U     "chromadb>=1.0,<2"     "sentence-transformers>=6.1,<7"     "transformers>=5.18,<6"     "datasets>=3,<5"     "ollama>=0.5,<1"     "gradio>=5,<7"     "fastapi>=0.115,<1"     "uvicorn>=0.30,<1"

if IN_COLAB:
    !rm -rf /content/vicare-rag
    !git clone -q https://github.com/humman0id/vicare-rag.git /content/vicare-rag
    %pip install -q --no-deps -e /content/vicare-rag
    if "/content/vicare-rag/src" not in sys.path:
        sys.path.insert(0, "/content/vicare-rag/src")

import sentence_transformers, transformers, vicare
print("sentence-transformers:", sentence_transformers.__version__)
print("transformers:", transformers.__version__)
print("✅ Đã nạp gói vicare từ:", vicare.__file__)


### 0.1. Cài đặt & khởi động Ollama server + tải LLM (`qwen2.5:3b`)


In [ ]:
import shutil, subprocess, time, requests

# qwen2.5:3b vừa gọn trên T4 miễn phí (15 GB VRAM) và công bằng với baseline.
LLM_MODEL = "qwen2.5:3b"
OLLAMA_URL = "http://127.0.0.1:11434"

if shutil.which("ollama") is None:
    if not IN_COLAB:
        raise RuntimeError("Hãy cài Ollama từ https://ollama.com rồi chạy lại.")
    print("Cài Ollama cho Google Colab...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)
    cmd = (
        "curl -fL --retry 3 https://ollama.com/download/ollama-linux-amd64.tar.zst "
        "| tar --zstd -xf - -C /usr/local"
    )
    subprocess.run(["bash", "-o", "pipefail", "-c", cmd], check=True)

def server_ready():
    try:
        return requests.get(OLLAMA_URL + "/api/tags", timeout=2).ok
    except requests.RequestException:
        return False

if not server_ready():
    print("Đang khởi động Ollama server...")
    log = open("/tmp/vicare_ollama.log", "w")
    proc = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(45):
        if server_ready() or proc.poll() is not None:
            break
        time.sleep(1)

if not server_ready():
    raise RuntimeError("Ollama server chưa sẵn sàng. Xem /tmp/vicare_ollama.log")

subprocess.run(["ollama", "pull", LLM_MODEL], check=True)
print("✅ Ollama server & model sẵn sàng:", LLM_MODEL)


## 1. Tải dữ liệu ViHealthQA & tạo tập `dev` cố định (100 câu, `seed=2026`)


In [ ]:
from vicare.data import load_vihealthqa
from vicare.chunking import build_chunks

corpus, dev, gold = load_vihealthqa(dev_seed=2026, dev_size=100)
chunks = build_chunks(corpus, chunk_size=200, overlap=40)

lens = sorted(len(d["text"].split()) for d in corpus)
print(f"Kho tài liệu: {len(corpus):,} văn bản → {len(chunks):,} chunk (CHUNK_SIZE=200, OVERLAP=40)")
print(f"Độ dài theo từ: trung vị {lens[len(lens)//2]}, tối đa {lens[-1]}")
print(f"Tập dev cố định: {len(dev)} câu")


## 2. Chạy Baseline mặc định (`intfloat/multilingual-e5-small`) để đối chứng công bằng


In [ ]:
from dataclasses import replace
from vicare.config import BASELINE_CONFIG
from vicare.embeddings import ViCareEmbedder
from vicare.retriever import ViCareRetriever
from vicare.rag import ViCareRAG
from vicare.evaluation import (
    evaluate_retrieval,
    evaluate_threshold_calibration,
    evaluate_generation,
    save_json_report,
)

N_CHUNKS = None  # None = toàn bộ 11,227 chunk (đánh giá cuối cùng); đặt 2000 nếu muốn thử nhanh.
N_JUDGE = 10     # Giữ nguyên số câu đánh giá giữa Baseline và Bản cải tiến (có thể nâng lên 20).

baseline_cfg = replace(BASELINE_CONFIG, n_chunks=N_CHUNKS, n_judge=N_JUDGE, llm_model=LLM_MODEL)
e5_embedder = ViCareEmbedder(baseline_cfg)
e5_retriever = ViCareRetriever(baseline_cfg, e5_embedder, chunks)
e5_rag = ViCareRAG(baseline_cfg, e5_retriever)

baseline_retrieval = evaluate_retrieval(e5_retriever, dev)
baseline_calibration = evaluate_threshold_calibration(e5_retriever, dev)
baseline_judge = evaluate_generation(e5_rag, dev, n_judge=N_JUDGE)
e5_telemetry = e5_embedder.telemetry()
e5_indexer_telemetry = e5_retriever.telemetry()

baseline_report = {
    "profile": baseline_cfg.profile_name,
    "settings": {
        "embedding_model": baseline_cfg.embedding_model,
        "chunk_size": baseline_cfg.chunk_size,
        "overlap": baseline_cfg.overlap,
        "n_chunks": baseline_cfg.n_chunks,
        "min_score": baseline_cfg.min_score,
        "llm_model": baseline_cfg.llm_model,
    },
    "embedder_telemetry": e5_telemetry,
    "indexer_telemetry": e5_indexer_telemetry,
    "retrieval": baseline_retrieval,
    "calibration": baseline_calibration,
    "judge": baseline_judge,
}
save_json_report("vicare_baseline.json", baseline_report)

# Giải phóng VRAM của model E5 trước khi nạp EmbeddingGemma 2 trên GPU T4 (15 GB).
e5_embedder.unload()

print("=== BASELINE (multilingual-e5-small) ===")
print("Telemetry:", e5_telemetry)
print("Indexer:", e5_indexer_telemetry)
print("Retrieval:", baseline_retrieval)
print("Judge:", baseline_judge)


## 3. Chạy Bản cải tiến (`google/embeddinggemma-2` + Prompt/Citation/Guardrails nâng cấp)

- Tải `google/embeddinggemma-2` ở chế độ **chỉ văn bản (271M)**; trên T4 miễn phí tự dùng **`float32`** an toàn (không dùng `float16`).
- Hiệu chỉnh `MIN_SCORE` tự động bằng phân bố điểm *trong kho* (`dev`) so với *ngoài kho* (`DEFAULT_OUT_OF_SCOPE_PROBES`).
- Bật **Improved Prompt + Strict `[k]` Citation Enforcement + Upgraded Emergency Guardrails**.


In [ ]:
from vicare.config import EMBEDDINGGEMMA2_CONFIG

improved_cfg_raw = replace(
    EMBEDDINGGEMMA2_CONFIG,
    n_chunks=N_CHUNKS,
    n_judge=N_JUDGE,
    llm_model=LLM_MODEL,
)
gemma_embedder = ViCareEmbedder(improved_cfg_raw)
gemma_retriever = ViCareRetriever(improved_cfg_raw, gemma_embedder, chunks)

improved_retrieval = evaluate_retrieval(gemma_retriever, dev)
improved_calibration = evaluate_threshold_calibration(gemma_retriever, dev)
calibrated_min_score = improved_calibration["recommended_min_score"]
print(f"✅ Ngưỡng MIN_SCORE hiệu chỉnh cho EmbeddingGemma 2: {calibrated_min_score:.4f}")
print("Phân bố điểm hiệu chỉnh:", improved_calibration)

improved_cfg = replace(improved_cfg_raw, min_score=calibrated_min_score)
gemma_rag = ViCareRAG(improved_cfg, gemma_retriever)
improved_judge = evaluate_generation(gemma_rag, dev, n_judge=N_JUDGE)
gemma_telemetry = gemma_embedder.telemetry()
gemma_indexer_telemetry = gemma_retriever.telemetry()

improved_report = {
    "profile": improved_cfg.profile_name,
    "settings": {
        "embedding_model": improved_cfg.embedding_model,
        "chunk_size": improved_cfg.chunk_size,
        "overlap": improved_cfg.overlap,
        "n_chunks": improved_cfg.n_chunks,
        "min_score": improved_cfg.min_score,
        "llm_model": improved_cfg.llm_model,
        "query_prompt_name": improved_cfg.query_prompt_name,
        "truncate_dim": improved_cfg.truncate_dim,
    },
    "embedder_telemetry": gemma_telemetry,
    "indexer_telemetry": gemma_indexer_telemetry,
    "retrieval": improved_retrieval,
    "calibration": improved_calibration,
    "judge": improved_judge,
}

print("\n=== IMPROVED (google/embeddinggemma-2 + Guardrails) ===")
print("Telemetry:", gemma_telemetry)
print("Indexer:", gemma_indexer_telemetry)
print("Retrieval:", improved_retrieval)
print("Judge:", improved_judge)


## 4. Bảng so sánh Trước / Sau & Phân tích Đánh đổi (Trade-offs & Caveats)


In [ ]:
from vicare.evaluation import format_comparison_table

print(format_comparison_table(baseline_report, improved_report))

print("\n--- SO SÁNH TÀI NGUYÊN & ĐÁNH ĐỔI (HONEST TRADE-OFFS) ---")
b_emb, i_emb = baseline_report["embedder_telemetry"], improved_report["embedder_telemetry"]
b_idx, i_idx = baseline_report["indexer_telemetry"], improved_report["indexer_telemetry"]
b_ret, i_ret = baseline_report["retrieval"], improved_report["retrieval"]

rows = [
    ("Model", b_emb["embedding_model"], i_emb["embedding_model"]),
    ("Params (M)", b_emb["param_count_millions"], i_emb["param_count_millions"]),
    ("Vector Dim", b_emb["embedding_dim"], i_emb["embedding_dim"]),
    ("Dtype", b_emb["dtype"], i_emb["dtype"]),
    ("Index Time (s)", b_idx["index_time_s"], i_idx["index_time_s"]),
    ("Avg Query (ms)", b_ret["avg_query_ms"], i_ret["avg_query_ms"]),
    ("Model VRAM (MB)", b_emb["model_vram_mb"], i_emb["model_vram_mb"]),
    ("MIN_SCORE", baseline_report["settings"]["min_score"], improved_report["settings"]["min_score"]),
]
print(f"{'Hạng mục':<18} {'Baseline (E5-small)':<32} {'Improved (EmbeddingGemma 2)':<32}")
print("-" * 82)
for label, v1, v2 in rows:
    print(f"{label:<18} {str(v1):<32} {str(v2):<32}")

save_json_report("vicare_comparison.json", {"baseline": baseline_report, "improved": improved_report})
print("\n✅ Đã lưu vicare_baseline.json và vicare_comparison.json")


### 4.1. Minh chứng định tính (Trong kho, Cấp cứu, Phủ định không cấp cứu, Ngoài kho)


In [ ]:
probe_questions = [
    ("TRONG KHO", "Đang chích ngừa viêm gan B có chích ngừa Covid-19 được không?"),
    ("CẤP CỨU", "Trẻ bị sốt cao kèm co giật thì phải làm gì?"),
    ("PHỦ ĐỊNH (không báo động giả)", "Tôi bị ho nhẹ vài ngày, không đau ngực và không khó thở, có cần đi khám không?"),
    ("NGOÀI PHẠM VI", "Thủ đô của nước Úc là gì?"),
]

for tag, q in probe_questions:
    res = gemma_rag.ask(q)
    top_score = round(res["hits"][0]["score"], 4) if res["hits"] else None
    print(f"=== [{tag}] {q} ===")
    print(res["answer"])
    print(f"Refused={res['refused']} ({res['refusal_reason']}) | Emergency={res['emergency_triggered']} {res['emergency_categories']} | TopScore={top_score}")
    print(f"CitationAudit={res['citation_audit']}")
    for s in res["sources"]:
        mark = "✓" if s["cited"] else " "
        print(f"  [{s['index']}]{mark} {s['url'] or s['doc_id']} (score={s['score']:.3f})")
    print()


## 5. Demo giao diện Gradio tương tác


In [ ]:
ENABLE_GRADIO = True  # Đổi thành False nếu chỉ muốn chạy benchmark.

if ENABLE_GRADIO:
    from vicare.ui import build_demo
    demo = build_demo(gemma_rag)
    demo.launch(share=IN_COLAB, debug=False, show_error=True)
else:
    print("Đã bỏ qua Gradio. Đặt ENABLE_GRADIO = True để mở demo.")
